
# Web Data Extraction & APIs — Live Notebook Lecture (90 minutes)

**Focus site:** `https://quotes.toscrape.com`  
**Goals:**
- Understand how web pages load: HTTP/HTTPS, requests, responses, HTML/CSS/JS, rendering.
- Use DevTools to inspect requests & responses.
- Reproduce a browser request via cURL and then with Python `requests`.
- Parse HTML with **BeautifulSoup** in multiple ways.
- Extract quotes (text), authors, and tags.
- Save results with **pandas** to a CSV (custom separators).
- Learn responsible scraping practices.
- (Optional) Brief note on using APIs where available.
- **Assignment**: choose your own site to navigate and scrape responsibly.

> You’ll run this notebook top-to-bottom during the lecture. Fill TODOs where asked.  
> Where you see “(paste image here)” you can drag & drop screenshots from DevTools into the notebook.



## Setup

We’ll rely on these packages:
- `requests`
- `beautifulsoup4`
- `pandas`

If you need to install them in your environment, run (uncomment if needed):

```bash
# !pip install requests beautifulsoup4 pandas
```



## 1) How Web Pages Work (HTTP/HTTPS → HTML → Render)

At a high level:
1. Your browser sends an **HTTP/HTTPS GET** request to a URL.  
2. The **server** responds with an **HTTP response**, typically HTML.  
3. The browser parses HTML and often issues **additional requests** for CSS, JS, fonts, and images.  
4. The browser **renders** the page incrementally; JavaScript may fetch more data (**XHR/fetch**) and modify the DOM.

```
You            Internet          Server
 |  GET /page -----> |  ...  | -----> Receives request
 | <-----  HTML      |  ...  | <----- Responds with HTML
 |  GET /style.css   |  ...  | -----> Additional requests (CSS/JS/images)
 |  GET /script.js   |  ...  |
 | <-----  CSS/JS    |  ...  |
[Render DOM + run JS; possibly fetch JSON via XHR/fetch]
```

**Static vs. dynamic** pages:  
- Static pages contain all key data directly in the HTML.  
- Dynamic pages may **render data via JavaScript** (e.g., JSON APIs). For those, you may:
  - Call the **same JSON endpoints** the page uses, or
  - Use a JS-capable tool (e.g., Playwright/Selenium) if needed.

> Our target site `quotes.toscrape.com` is convenient for learning: it’s simple and mostly static.



## 2) DevTools: Inspect Requests/Responses

Open your browser’s **Developer Tools** → **Network** tab, then reload `https://quotes.toscrape.com/`.

What to look for:
- **Request URL**: `https://quotes.toscrape.com/`
- **Method**: GET
- **Status**: 200 OK
- **Response headers**: content type, encoding, etc.
- **Request headers**: `User-Agent`, `Accept`, `Accept-Language`, etc.
- **Other requests**: CSS, images, possibly additional pages when clicking pagination.

*(paste your screenshot(s) below)*



## 3) “Copy as cURL”

From DevTools, you can right-click the request and **Copy → Copy as cURL**.  
This reproduces the browser’s request on the command line.

### Example
```bash
curl 'https://quotes.toscrape.com/' \
  --compressed \
  -H 'User-Agent: Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:143.0) Gecko/20100101 Firefox/143.0' \
  -H 'Accept: text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8' \
  -H 'Accept-Language: en-GB,en;q=0.5' \
  -H 'Accept-Encoding: gzip, deflate, br, zstd' \
  -H 'Connection: keep-alive' \
  -H 'Upgrade-Insecure-Requests: 1' \
  -H 'Sec-Fetch-Dest: document' \
  -H 'Sec-Fetch-Mode: navigate' \
  -H 'Sec-Fetch-Site: none' \
  -H 'Sec-Fetch-User: ?1' \
  -H 'Priority: u=0, i' \
  -H 'Pragma: no-cache' \
  -H 'Cache-Control: no-cache'
```

> Try running it in a terminal or a shell-enabled notebook with `!curl ...`.  
> You should receive HTML for the homepage.


In [ ]:

# OPTIONAL: Try running the cURL command directly from a notebook cell (requires curl to be installed).
# Remove the leading '#' characters and run if your environment supports it.

# !curl 'https://quotes.toscrape.com/' #   --compressed #   -H 'User-Agent: Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:143.0) Gecko/20100101 Firefox/143.0' #   -H 'Accept: text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8' #   -H 'Accept-Language: en-GB,en;q=0.5' #   -H 'Accept-Encoding: gzip, deflate, br, zstd' #   -H 'Connection: keep-alive' #   -H 'Upgrade-Insecure-Requests: 1' #   -H 'Sec-Fetch-Dest: document' #   -H 'Sec-Fetch-Mode: navigate' #   -H 'Sec-Fetch-Site: none' #   -H 'Sec-Fetch-User': '?1' #   -H 'Priority: u=0, i' #   -H 'Pragma: no-cache' #   -H 'Cache-Control: no-cache'



## 4) Reproducing the Request in Python (`requests`)

Below we mirror the important headers and fetch the HTML.  
We’ll write a small helper that returns the **HTML as a string**.


In [ ]:

from __future__ import annotations
from typing import Optional, Dict

import requests

DEFAULT_HEADERS: Dict[str, str] = {
    "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:143.0) Gecko/20100101 Firefox/143.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-GB,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
    "Priority": "u=0, i",
    "Pragma": "no-cache",
    "Cache-Control": "no-cache",
}

def fetch_html(url: str, headers: Optional[Dict[str, str]] = None, timeout_s: float = 15.0) -> str:
    """
    Fetch HTML for a given URL using GET.

    Args:
        url: Absolute URL to fetch.
        headers: Optional mapping of HTTP headers to send.
        timeout_s: Socket timeout in seconds.
    Returns:
        The response body decoded as text (requests tries to detect encoding).
    Raises:
        requests.HTTPError if the status is 4xx or 5xx.
    """
    merged_headers: Dict[str, str] = {**DEFAULT_HEADERS, **(headers or {})}
    resp = requests.get(url, headers=merged_headers, timeout=timeout_s)
    resp.raise_for_status()
    return resp.text

# Demo: fetch the homepage (prints first 500 characters).
if __name__ == "__main__":
    try:
        html_preview: str = fetch_html("https://quotes.toscrape.com/")
        print(html_preview[:500])
    except Exception as e:
        print(f"Fetch failed (likely due to offline/no-internet environment): {e}")



## 5) Parsing with BeautifulSoup — Quotes Only

We’ll show **three approaches** to extract the **quote text** from the homepage:
1. **CSS selectors** (`select` / `select_one`)
2. **Tag & class** via `find_all`
3. **Direct iteration** over containers + element lookup

Each function will return `list[str]` (just the quotes). Make sure you’ve fetched HTML first.


In [ ]:

from __future__ import annotations
from typing import List
from bs4 import BeautifulSoup

def quotes_via_css(html: str) -> List[str]:
    """
    Extract quotes using CSS selectors.

    Args:
        html: Page HTML as a string.
    Returns:
        List of quote texts.
    """
    soup = BeautifulSoup(html, "html.parser")
    return [el.get_text(strip=True) for el in soup.select("div.quote span.text")]

def quotes_via_find_all(html: str) -> List[str]:
    """
    Extract quotes using find_all with tag/class.

    Args:
        html: Page HTML as a string.
    Returns:
        List of quote texts.
    """
    soup = BeautifulSoup(html, "html.parser")
    quotes: List[str] = []
    for qdiv in soup.find_all("div", class_="quote"):
        text_span = qdiv.find("span", class_="text")
        if text_span:
            quotes.append(text_span.get_text(strip=True))
    return quotes

def quotes_via_iteration(html: str) -> List[str]:
    """
    Extract quotes by iterating containers and then locating inner spans.

    Args:
        html: Page HTML as a string.
    Returns:
        List of quote texts.
    """
    soup = BeautifulSoup(html, "html.parser")
    quotes: List[str] = []
    for container in soup.select("div.quote"):
        span = container.select_one("span.text")
        if span:
            quotes.append(span.get_text(strip=True))
    return quotes

# Demo run (safe if offline; wrapped in try/except)
if __name__ == "__main__":
    try:
        page_html: str = fetch_html("https://quotes.toscrape.com/")
        print("CSS:", quotes_via_css(page_html)[:3])
        print("find_all:", quotes_via_find_all(page_html)[:3])
        print("iteration:", quotes_via_iteration(page_html)[:3])
    except Exception as e:
        print(f"Skipping demo due to environment: {e}")



## 6) More Parsing — Quotes, Authors, Tags

Now we’ll extract a **tuple** for each quote:  
`(quote_text: str, author: str, tags: list[str])`

We’ll also show how to **follow pagination** (clicking “Next”) to get multiple pages.


In [ ]:

from __future__ import annotations
from typing import List, Tuple, Optional
from bs4 import BeautifulSoup
from urllib.parse import urljoin

QuoteRow = Tuple[str, str, List[str]]  # (quote, author, tags)

def parse_quotes_page(html: str) -> List[QuoteRow]:
    """
    Parse a quotes.toscrape.com HTML page into a list of (quote, author, tags).

    Args:
        html: Page HTML as a string.
    Returns:
        A list of tuples (quote_text, author, tags_list).
    """
    soup = BeautifulSoup(html, "html.parser")
    rows: List[QuoteRow] = []
    for q in soup.select("div.quote"):
        text_el = q.select_one("span.text")
        author_el = q.select_one("small.author")
        tag_els = q.select("div.tags a.tag")
        if not text_el or not author_el:
            continue
        text: str = text_el.get_text(strip=True)
        author: str = author_el.get_text(strip=True)
        tags: List[str] = [t.get_text(strip=True) for t in tag_els]
        rows.append((text, author, tags))
    return rows

def find_next_page_url(html: str, base_url: str) -> Optional[str]:
    """
    Find the absolute URL for the 'Next' page, if present.

    Args:
        html: Current page HTML.
        base_url: The URL of the current page (used to resolve relative links).
    Returns:
        Absolute URL to the next page, or None if there is no next page.
    """
    soup = BeautifulSoup(html, "html.parser")
    next_link = soup.select_one("li.next a")
    if next_link and next_link.get("href"):
        return urljoin(base_url, next_link["href"])
    return None

def scrape_all_quotes(start_url: str, max_pages: int = 10) -> List[QuoteRow]:
    """
    Crawl forward from start_url by following the 'Next' link up to max_pages.

    Args:
        start_url: The first page URL to scrape.
        max_pages: Safety cap on number of pages to walk.
    Returns:
        A list of (quote, author, tags) collected across pages.
    """
    all_rows: List[QuoteRow] = []
    url: Optional[str] = start_url
    pages_visited: int = 0

    while url and pages_visited < max_pages:
        html: str = fetch_html(url)
        page_rows: List[QuoteRow] = parse_quotes_page(html)
        all_rows.extend(page_rows)
        url = find_next_page_url(html, base_url=url)
        pages_visited += 1
    return all_rows

# Demo (wrapped for offline safety)
if __name__ == "__main__":
    try:
        rows = scrape_all_quotes("https://quotes.toscrape.com/", max_pages=3)
        print(f"Collected {len(rows)} rows (first 2 shown):")
        for r in rows[:2]:
            print(r)
    except Exception as e:
        print(f"Scrape demo skipped: {e}")



## 7) Save Results with `pandas` (CSV)

We’ll convert the tuples to a DataFrame and then **save to CSV** with:
- CSV **separator**: `;`
- **Tags** field joined by commas: `","`

Resulting CSV columns: `quote;author;tags`


In [ ]:

from __future__ import annotations
from typing import List, Tuple
import pandas as pd

def rows_to_dataframe(rows: List[Tuple[str, str, List[str]]]) -> pd.DataFrame:
    """
    Convert parsed rows to a pandas DataFrame with columns: quote, author, tags.

    Args:
        rows: List of (quote, author, tags) tuples.
    Returns:
        A pandas DataFrame.
    """
    records = [{"quote": q, "author": a, "tags": ",".join(t)} for (q, a, t) in rows]
    return pd.DataFrame.from_records(records, columns=["quote", "author", "tags"])

def save_quotes_csv(rows: List[Tuple[str, str, List[str]]], path: str) -> None:
    """
    Save rows to CSV with ';' separator and tags joined by commas.

    Args:
        rows: List of (quote, author, tags) tuples.
        path: Output file path.
    Returns:
        None
    """
    df = rows_to_dataframe(rows)
    df.to_csv(path, sep=";", index=False)

# Demo save (skips if no network to fetch rows)
if __name__ == "__main__":
    try:
        rows = scrape_all_quotes("https://quotes.toscrape.com/", max_pages=3)
        save_path = "quotes_sample.csv"
        save_quotes_csv(rows, save_path)
        print(f"Saved {len(rows)} rows to {save_path}")
        display(rows_to_dataframe(rows).head(3))
    except Exception as e:
        print(f"Save demo skipped: {e}")



## 8) Responsible Scraping — Notes & Best Practices

- **Check robots.txt** and the site’s **Terms of Service**.
- **Be gentle**: add small delays between requests; don’t hammer servers.
- **Identify yourself**: set a sensible `User-Agent`. Consider contact info for research use.
- **Limit scope**: only scrape what you need; set **max pages**.
- **Avoid PII** and restricted data.
- **Cache** responses where appropriate to reduce load.
- **Respect rate limits** if using APIs; use API keys securely.
- **Legal & ethical**: laws vary by jurisdiction; when in doubt, consult your organization’s policy.



### (Optional) APIs vs. Scraping

If a site offers a **documented API**, prefer it:
- Stable schemas (JSON), explicit rate limits, authentication, usage terms.
- Often easier and more reliable than parsing HTML.

When no API exists or data is presentation-only, scraping may be the only option—follow the best practices above.



## 🔍 Assignment — Choose Your Own List Page & Navigate It

**Your task:** Pick a website that lists items across **multiple pages** (e.g., quotes, articles, products, events).  
Avoid login walls and sites that forbid scraping. Educational sandboxes like `books.toscrape.com` are fine, but feel free to choose your own.

### Requirements
1. **Describe the target** (1–2 sentences): what you’re scraping and why it’s suitable.
2. **Inspect with DevTools**: include 1–2 screenshots highlighting the key request and the HTML structure you’ll parse.
3. **Fetcher**: implement a typed function  
   `fetch_html(url: str, headers: Optional[Dict[str, str]] = None, timeout_s: float = 15.0) -> str`  
   (you may reuse ours).
4. **Parser**: implement a typed function that extracts at least **three fields** per item (e.g., title, author, date/tags).
   - Return type: `list[tuple[str, str, list[str]]]` or a `dataclass`. If not applicable, use `list[dict[str, Any]]`.
5. **Pagination**: follow “Next” (or numbered pages) until you collect **≥ 50 items** (or exhaust pages).
6. **CSV export**: save to a CSV with **`;`** as the separator; if you have a list field, join with **`,`**.
7. **Politeness**: add a small `time.sleep` between requests; keep a `max_pages` cap.
8. **Documentation**: brief markdown section summarizing **what worked**, **what broke**, and **how you handled it**.

### Deliverables
- This notebook with completed cells and your screenshots.
- A CSV file in the repo (or alongside the notebook).

### Grading (guide)
- Correctness (40%): code runs, types used, CSV created.
- Engineering (30%): pagination, error-handling, clean functions, docstrings.
- Analysis (20%): clear explanation & DevTools evidence.
- Responsibility (10%): rate limiting, scope limits, and ToS awareness.

> Tip: If your chosen site changes frequently, cache the first page locally while developing.



## ✏️ Exercise Cells (for assignment)

Use the following skeleton to implement your selected target scraper.


In [ ]:

# TODO: Replace with your chosen URL
TARGET_URL: str = "https://example.com/list-page"
MAX_PAGES: int = 10

# TODO: Add any site-specific headers if needed
CUSTOM_HEADERS: dict[str, str] = {}

# You can reuse fetch_html from earlier (imported in this notebook).


In [ ]:

from __future__ import annotations
from typing import Optional, Dict, List, Tuple, Any
from dataclasses import dataclass
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin

@dataclass
class Item:
    """Example data model. Modify fields for your target site."""
    title: str
    meta: str
    tags: List[str]

def parse_items_page(html: str) -> List[Item]:
    """
    TODO: Implement parsing for your chosen site.

    Args:
        html: HTML content for one listing page.
    Returns:
        A list of Item objects.
    """
    soup = BeautifulSoup(html, "html.parser")
    items: List[Item] = []
    # TODO: Replace 'div.item' etc. with actual selectors for your site.
    for el in soup.select("div.item"):
        title_el = el.select_one(".title")
        meta_el = el.select_one(".meta")
        tag_els = el.select(".tag")
        if not title_el or not meta_el:
            continue
        title: str = title_el.get_text(strip=True)
        meta: str = meta_el.get_text(strip=True)
        tags: List[str] = [t.get_text(strip=True) for t in tag_els]
        items.append(Item(title=title, meta=meta, tags=tags))
    return items

def find_next_url(html: str, base_url: str) -> Optional[str]:
    """
    TODO: Implement pagination discovery for your chosen site.
    """
    soup = BeautifulSoup(html, "html.parser")
    nxt = soup.select_one("a.next")
    return urljoin(base_url, nxt["href"]) if nxt and nxt.get("href") else None

def scrape_items(start_url: str, max_pages: int = 10, delay_s: float = 1.0) -> List[Item]:
    """
    Crawl pages by following pagination links.
    """
    items: List[Item] = []
    url: Optional[str] = start_url
    pages: int = 0
    while url and pages < max_pages:
        html = fetch_html(url, headers=CUSTOM_HEADERS or None)
        page_items = parse_items_page(html)
        items.extend(page_items)
        url = find_next_url(html, base_url=url)
        pages += 1
        time.sleep(delay_s)
    return items


In [ ]:

import pandas as pd

def items_to_dataframe(items: List[Item]) -> pd.DataFrame:
    """
    Convert Item list to DataFrame with tags joined by commas.
    """
    return pd.DataFrame([{"title": it.title, "meta": it.meta, "tags": ",".join(it.tags)} for it in items])

def save_items_csv(items: List[Item], path: str) -> None:
    """
    Save items to CSV with ';' separator.
    """
    df = items_to_dataframe(items)
    df.to_csv(path, sep=";", index=False)

# Demo-run template (uncomment when your TARGET_URL is set)
# items = scrape_items(TARGET_URL, max_pages=MAX_PAGES, delay_s=1.0)
# save_items_csv(items, "assignment_items.csv")
# items_to_dataframe(items).head()



## Wrap-up

You learned how to:
- Inspect & reproduce HTTP requests (DevTools → cURL → Python).
- Parse HTML with BeautifulSoup using multiple strategies.
- Extract structured data and save it to CSV with custom separators.
- Scrape responsibly.

Happy scraping! 🕸️
